<small><b>03 Classification</b> — Add filter facets: reuse topics/flags as <code>product_type</code>, then zero-shot (<code>facebook/bart-large-mnli</code>) for <code>target_user</code> / <code>pricing_model</code> / <code>core_function</code>. Write <code>data/saas_enriched.csv</code>.</small>

<small><b>Step 0 — Setup</b>. First zero-shot run downloads BART (~1.6GB). On CPU use a small <code>SAMPLE_SIZE</code> first.</small>

In [2]:
from pathlib import Path
import os
import sys

from dotenv import load_dotenv

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
load_dotenv(ROOT / ".env", override=True)

from src.classify import (
    CLEANED_PATH,
    ENRICHED_PATH,
    LABEL_SETS,
    ZERO_SHOT_MODEL,
    enrich,
    filter_enriched,
    load_cleaned,
    product_type_from_flags,
)

# Start small on CPU (set None for full dataset — slow)
SAMPLE_SIZE = 40
USE_ZERO_SHOT = True

print("CLEANED_PATH:", CLEANED_PATH)
print("ENRICHED_PATH:", ENRICHED_PATH)
print("ZERO_SHOT_MODEL:", ZERO_SHOT_MODEL)
print("LABEL_SETS:", LABEL_SETS)
print("SAMPLE_SIZE:", SAMPLE_SIZE)
print("HF_TOKEN set:", bool(os.getenv("HF_TOKEN")))

CLEANED_PATH: C:\Users\suxia\Desktop\Saas-Recommender(2026)\data\saas_cleaned.csv
ENRICHED_PATH: C:\Users\suxia\Desktop\Saas-Recommender(2026)\data\saas_enriched.csv
ZERO_SHOT_MODEL: facebook/bart-large-mnli
LABEL_SETS: {'target_user': ['Solo Founder', 'SMB', 'Enterprise'], 'pricing_model': ['Free', 'Freemium', 'Paid'], 'core_function': ['Analytics', 'Automation', 'Collaboration', 'AI Agent', 'Developer Tool']}
SAMPLE_SIZE: 40
HF_TOKEN set: True


<small><b>Step 1 — Load cleaned data</b> from notebook 01.</small>

In [3]:
df = load_cleaned(CLEANED_PATH, sample_size=SAMPLE_SIZE)
print("rows:", len(df))
display(df[["name", "tagline", "topics", "main_category", "is_ai_product", "is_saas_product", "is_dev_tool", "is_productivity"]].head(5))

rows: 40


,name,tagline,topics,main_category,is_ai_product,is_saas_product,is_dev_tool,is_productivity
0,Bluedot 2.1,Record on Apple Watch. Sync with Claude,Productivity,Productivity,0,0,0,1
1,Powabase,"Build AI apps with Postgres, RAG, and agents","AI, Developer Tools",AI,1,0,1,0
2,Oasis Browser for Mac,A privacy-first AI browser you can train anony...,"AI, Productivity",AI,1,0,0,1
3,zero.xyz,"Give your AI agent access to ~8k tools, APIs a...","AI, Productivity",AI,1,0,0,1
4,Coworker AI,More AI for less spend with context-aware mode...,"AI, SaaS, Productivity",SaaS,1,1,0,1


<small><b>Step 2 — product_type from existing flags</b> (SaaS / AI / Developer Tools / Productivity). No model needed.</small>

In [4]:
df["product_type"] = df.apply(product_type_from_flags, axis=1)
print(df["product_type"].value_counts())
display(df[["name", "topics", "main_category", "product_type"]].head(8))

product_type
AI                 15
Developer Tools     9
General             7
Productivity        5
SaaS                4
Name: count, dtype: int64


,name,topics,main_category,product_type
0,Bluedot 2.1,Productivity,Productivity,Productivity
1,Powabase,"AI, Developer Tools",AI,AI
2,Oasis Browser for Mac,"AI, Productivity",AI,AI
3,zero.xyz,"AI, Productivity",AI,AI
4,Coworker AI,"AI, SaaS, Productivity",SaaS,SaaS
5,Octolane,General,General,General
6,Mojito,Developer Tools,Developer Tools,Developer Tools
7,Layers,Developer Tools,Developer Tools,Developer Tools


<small><b>Step 3 — Zero-shot labels</b> with <code>facebook/bart-large-mnli</code>: Target User / Pricing Model / Core Function. Progress prints every 10 rows.</small>

In [5]:
print("Starting enrich (first run downloads the MNLI model)...", flush=True)
enriched = enrich(df, use_zero_shot=USE_ZERO_SHOT, show_progress=True)
cols = [
    "name", "product_type",
    "target_user", "target_user_score",
    "pricing_model", "pricing_model_score",
    "core_function", "core_function_score",
]
display(enriched[cols].head(10))
print("\ntarget_user:"); display(enriched["target_user"].value_counts())
print("pricing_model:"); display(enriched["pricing_model"].value_counts())
print("core_function:"); display(enriched["core_function"].value_counts())

Starting enrich (first run downloads the MNLI model)...


Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]

[classify] 1/40: Bluedot 2.1
[classify] 10/40: Pawse.ai
[classify] 20/40: Extend
[classify] 30/40: AgenticCalling AI
[classify] 40/40: Syncaut


,name,product_type,target_user,target_user_score,pricing_model,pricing_model_score,core_function,core_function_score
0,Bluedot 2.1,Productivity,Solo Founder,0.5558,Freemium,0.4119,Automation,0.3878
1,Powabase,AI,Solo Founder,0.5899,Paid,0.6061,Developer Tool,0.3753
2,Oasis Browser for Mac,AI,Solo Founder,0.6130,Paid,0.6032,AI Agent,0.2890
3,zero.xyz,AI,Solo Founder,0.4233,Free,0.8864,AI Agent,0.6043
4,Coworker AI,SaaS,Solo Founder,0.4322,Paid,0.7062,Collaboration,0.4575
5,Octolane,General,Solo Founder,0.5840,Paid,0.5900,Automation,0.4731
6,Mojito,Developer Tools,Solo Founder,0.4434,Free,0.9645,Automation,0.6443
7,Layers,Developer Tools,Solo Founder,0.5211,Free,0.9752,Developer Tool,0.3384
8,Calling Skills for AI Agents,AI,Solo Founder,0.4622,Paid,0.4085,AI Agent,0.7499
9,Pawse.ai,AI,Solo Founder,0.5447,Paid,0.7636,Automation,0.3657



target_user:


target_user
Solo Founder    38
SMB              1
Enterprise       1
Name: count, dtype: int64

pricing_model:


pricing_model
Paid        26
Free        11
Freemium     3
Name: count, dtype: int64

core_function:


core_function
Automation        14
Developer Tool    13
AI Agent          10
Collaboration      3
Name: count, dtype: int64

<small><b>Step 4 — Semantic recall + facet filter demo</b>. Example: keep AI Agent + Freemium + Solo Founder style filters on the enriched table.</small>

In [6]:
filtered = filter_enriched(
    enriched,
    product_type="All",
    target_user="Solo Founder",
    pricing_model="Freemium",
    core_function="AI Agent",
)
print("filter: Solo Founder + Freemium + AI Agent →", len(filtered), "rows")
display(filtered[cols].head(10))

# Relaxed example if the strict combo is empty on a small sample
if filtered.empty:
    relaxed = filter_enriched(enriched, core_function="AI Agent")
    print("relaxed filter core_function=AI Agent →", len(relaxed), "rows")
    display(relaxed[cols].head(10))

filter: Solo Founder + Freemium + AI Agent → 0 rows


,name,product_type,target_user,target_user_score,pricing_model,pricing_model_score,core_function,core_function_score


relaxed filter core_function=AI Agent → 10 rows


,name,product_type,target_user,target_user_score,pricing_model,pricing_model_score,core_function,core_function_score
0,Oasis Browser for Mac,AI,Solo Founder,0.6130,Paid,0.6032,AI Agent,0.2890
1,zero.xyz,AI,Solo Founder,0.4233,Free,0.8864,AI Agent,0.6043
2,Calling Skills for AI Agents,AI,Solo Founder,0.4622,Paid,0.4085,AI Agent,0.7499
3,Krater,AI,Solo Founder,0.5689,Paid,0.4844,AI Agent,0.7873
4,baz.studio,General,Solo Founder,0.4037,Paid,0.4561,AI Agent,0.5829
5,BobCA,AI,Solo Founder,0.5802,Paid,0.5787,AI Agent,0.7814
6,Chunk sidecars,AI,Solo Founder,0.5183,Free,0.9785,AI Agent,0.5039
7,AgenticCalling AI,AI,Solo Founder,0.4763,Paid,0.4723,AI Agent,0.6522
8,Fundraisly,AI,Solo Founder,0.5752,Paid,0.6821,AI Agent,0.8231
9,Kakunin,SaaS,Solo Founder,0.5328,Paid,0.6200,AI Agent,0.5118


<small><b>Step 5 — Export</b> <code>data/saas_enriched.csv</code> for Gradio filters / later ranking.</small>

In [7]:
ENRICHED_PATH.parent.mkdir(parents=True, exist_ok=True)
enriched.to_csv(ENRICHED_PATH, index=False)
print(f"Wrote {ENRICHED_PATH} -> {enriched.shape[0]} rows, {enriched.shape[1]} cols", flush=True)

Wrote C:\Users\suxia\Desktop\Saas-Recommender(2026)\data\saas_enriched.csv -> 40 rows, 43 cols
